In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
train = pd.read_csv("../data/application_train.csv")
test = pd.read_csv("../data/application_test.csv")
bureau_balance = pd.read_csv("../data/bureau_balance.csv")
bureau = pd.read_csv("../data/bureau.csv")
credut_balance = pd.read_csv("../data/credit_card_balance.csv")
installments_payments = pd.read_csv("../data/installments_payments.csv")
pos_cash = pd.read_csv("../data/POS_CASH_balance.csv")
previous_application = pd.read_csv("../data/previous_application.csv")

In [3]:
bureau_balance

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
...,...,...,...
27299920,5041336,-47,X
27299921,5041336,-48,X
27299922,5041336,-49,X
27299923,5041336,-50,X


In [4]:
bureau_balance['STATUS'].value_counts()

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

In [5]:
from sklearn.preprocessing import OrdinalEncoder
ord_rank = ["C","X",'0','1','2','3','4','5']
encoder = OrdinalEncoder(categories= [ord_rank])
bureau_balance["encoded_status"] = encoder.fit_transform(bureau_balance[['STATUS']])

In [6]:
bb_aggregations = bureau_balance.groupby("SK_ID_BUREAU").agg({
    'MONTHS_BALANCE' : ["min" , 'max','size'],
    'encoded_status': ["mean" , 'max' , 'std']}
    )


In [7]:
bb_aggregations.columns = ['_'.join(col).strip() for col in bb_aggregations.columns.values]

bb_aggregations = bb_aggregations.reset_index()

bb_aggregations.head()

,SK_ID_BUREAU,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_size,encoded_status_mean,encoded_status_max,encoded_status_std
0,5001709,-96,0,97,0.113402,1.0,0.318731
1,5001710,-82,0,83,0.481928,2.0,0.612102
2,5001711,-3,0,4,1.750000,2.0,0.500000
3,5001712,-18,0,19,1.052632,2.0,1.025978
4,5001713,-21,0,22,1.000000,1.0,0.000000


In [8]:
merged_bureau = pd.merge(bureau, bb_aggregations, on='SK_ID_BUREAU', how='left')

In [9]:
merged_bureau

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_size,encoded_status_mean,encoded_status_max,encoded_status_std
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,0.0,Consumer credit,-131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,0.0,Credit card,-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,0.0,Consumer credit,-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,0.0,Credit card,-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,0.0,Consumer credit,-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.0,NaN,0.0,0,...,0.0,Microloan,-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.0,-2493.0,5476.5,0,...,0.0,Consumer credit,-2493,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.0,-970.0,NaN,0,...,0.0,Consumer credit,-967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.0,-1513.0,NaN,0,...,0.0,Consumer credit,-1508,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
num_cols = merged_bureau.select_dtypes(include = [np.number]).columns.tolist()

In [11]:
bureau_agg_dict = {}
num_cols.remove('SK_ID_CURR')
if 'SK_ID_BUREAU' in num_cols:
    num_cols.remove('SK_ID_BUREAU')
num_cols
        

['DAYS_CREDIT',
 'CREDIT_DAY_OVERDUE',
 'DAYS_CREDIT_ENDDATE',
 'DAYS_ENDDATE_FACT',
 'AMT_CREDIT_MAX_OVERDUE',
 'CNT_CREDIT_PROLONG',
 'AMT_CREDIT_SUM',
 'AMT_CREDIT_SUM_DEBT',
 'AMT_CREDIT_SUM_LIMIT',
 'AMT_CREDIT_SUM_OVERDUE',
 'DAYS_CREDIT_UPDATE',
 'AMT_ANNUITY',
 'MONTHS_BALANCE_min',
 'MONTHS_BALANCE_max',
 'MONTHS_BALANCE_size',
 'encoded_status_mean',
 'encoded_status_max',
 'encoded_status_std']

In [12]:
for col in num_cols:
    bureau_agg_dict[col] = ["min",'max','mean','std']

In [13]:
client_bureau_agg = merged_bureau.groupby('SK_ID_CURR').agg(bureau_agg_dict).reset_index()

client_bureau_agg.columns = ['_'.join(col).strip() if col[0] != 'SK_ID_CURR' else col[0] for col in client_bureau_agg.columns.values]

client_bureau_agg.head()

,SK_ID_CURR,DAYS_CREDIT_min,DAYS_CREDIT_max,DAYS_CREDIT_mean,DAYS_CREDIT_std,CREDIT_DAY_OVERDUE_min,CREDIT_DAY_OVERDUE_max,CREDIT_DAY_OVERDUE_mean,CREDIT_DAY_OVERDUE_std,DAYS_CREDIT_ENDDATE_min,...,encoded_status_mean_mean,encoded_status_mean_std,encoded_status_max_min,encoded_status_max_max,encoded_status_max_mean,encoded_status_max_std,encoded_status_std_min,encoded_status_std_max,encoded_status_std_mean,encoded_status_std_std
0,100001,-1572,-49,-735.000000,489.942514,0,0,0.0,0.0,-1329.0,...,0.910448,0.806279,2.0,3.0,2.142857,0.377964,0.00000,0.707107,0.483151,0.248221
1,100002,-1437,-103,-874.000000,431.451040,0,0,0.0,0.0,-1072.0,...,1.742898,0.616499,2.0,3.0,2.750000,0.462910,0.57735,1.062623,0.859489,0.228384
2,100003,-2586,-606,-1400.750000,909.826128,0,0,0.0,0.0,-2434.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,-1326,-408,-867.000000,649.124025,0,0,0.0,0.0,-595.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,-373,-62,-190.666667,162.297053,0,0,0.0,0.0,-128.0,...,1.606838,0.426238,2.0,2.0,2.000000,0.000000,0.00000,0.987096,0.521482,0.495914


In [14]:
merged_bureau_train = pd.merge(train, client_bureau_agg, on='SK_ID_CURR', how='left')

In [15]:
merged_bureau_train

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,encoded_status_mean_mean,encoded_status_mean_std,encoded_status_max_min,encoded_status_max_max,encoded_status_max_mean,encoded_status_max_std,encoded_status_std_min,encoded_status_std_max,encoded_status_std_mean,encoded_status_std_std
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,1.742898,0.616499,2.0,3.0,2.75,0.46291,0.577350,1.062623,0.859489,0.228384
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0.945228,0.342070,2.0,2.0,2.00,0.00000,0.508977,0.990275,0.869951,0.240649
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0.432432,NaN,2.0,2.0,2.00,NaN,0.834684,0.834684,0.834684,NaN


In [16]:
print("Train shape before:", train.shape)
print("Train shape after:", merged_bureau_train.shape)

Train shape before: (307511, 122)
Train shape after: (307511, 194)


In [17]:
merged_bureau_test = pd.merge(test, client_bureau_agg, on='SK_ID_CURR', how='left')

In [18]:
print("Train shape before:", test.shape)
print("Train shape after:", merged_bureau_test.shape)

Train shape before: (48744, 121)
Train shape after: (48744, 193)


In [19]:
credut_balance

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3840307,1036507,328243,-9,0.000,45000,NaN,0.0,NaN,NaN,0.000,...,0.000,0.000,NaN,0,NaN,NaN,0.0,Active,0,0
3840308,1714892,347207,-9,0.000,45000,0.0,0.0,0.0,0.0,0.000,...,0.000,0.000,0.0,0,0.0,0.0,23.0,Active,0,0
3840309,1302323,215757,-9,275784.975,585000,270000.0,270000.0,0.0,0.0,2250.000,...,273093.975,273093.975,2.0,2,0.0,0.0,18.0,Active,0,0
3840310,1624872,430337,-10,0.000,450000,NaN,0.0,NaN,NaN,0.000,...,0.000,0.000,NaN,0,NaN,NaN,0.0,Active,0,0


In [20]:
num_cols_c = credut_balance.select_dtypes(include= [np.number]).columns.tolist()

In [21]:
num_cols_c.remove('SK_ID_PREV')
if 'SK_ID_CURR' in num_cols_c:
    num_cols_c.remove('SK_ID_CURR')

In [22]:
credut_balance_agg_dict = {}
for col in num_cols_c:
    credut_balance_agg_dict[col] = ["min",'max','std','mean']

In [23]:
credut_balance_merge = credut_balance.groupby('SK_ID_CURR').agg(credut_balance_agg_dict).reset_index()

In [24]:
credut_balance_merge.columns = ['_'.join(col).strip() if col[0] != 'SK_ID_CURR' else col[0] for col in credut_balance_merge.columns.values]
credut_balance_merge.head()

,SK_ID_CURR,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_std,MONTHS_BALANCE_mean,AMT_BALANCE_min,AMT_BALANCE_max,AMT_BALANCE_std,AMT_BALANCE_mean,AMT_CREDIT_LIMIT_ACTUAL_min,...,CNT_INSTALMENT_MATURE_CUM_std,CNT_INSTALMENT_MATURE_CUM_mean,SK_DPD_min,SK_DPD_max,SK_DPD_std,SK_DPD_mean,SK_DPD_DEF_min,SK_DPD_DEF_max,SK_DPD_DEF_std,SK_DPD_DEF_mean
0,100006,-6,-1,1.870829,-3.5,0.0,0.00,0.000000,0.000000,270000,...,0.000000,0.000000,0,0,0.000000,0.000000,0,0,0.000000,0.000000
1,100011,-75,-2,21.505813,-38.5,0.0,189000.00,68127.238270,54482.111149,90000,...,10.288236,25.767123,0,0,0.000000,0.000000,0,0,0.000000,0.000000
2,100013,-96,-1,27.856777,-48.5,0.0,161420.22,43237.406997,18159.919219,45000,...,5.852328,18.719101,0,1,0.102062,0.010417,0,1,0.102062,0.010417
3,100021,-18,-2,5.049752,-10.0,0.0,0.00,0.000000,0.000000,675000,...,0.000000,0.000000,0,0,0.000000,0.000000,0,0,0.000000,0.000000
4,100023,-11,-4,2.449490,-7.5,0.0,0.00,0.000000,0.000000,45000,...,0.000000,0.000000,0,0,0.000000,0.000000,0,0,0.000000,0.000000


In [25]:
installments_payments

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
...,...,...,...,...,...,...,...,...
13605396,2186857,428057,0.0,66,-1624.0,NaN,67.500,NaN
13605397,1310347,414406,0.0,47,-1539.0,NaN,67.500,NaN
13605398,1308766,402199,0.0,43,-7.0,NaN,43737.435,NaN
13605399,1062206,409297,0.0,43,-1986.0,NaN,67.500,NaN


In [26]:
num_cols_ins = installments_payments.select_dtypes(include= [np.number]).columns.tolist()

num_cols_ins.remove('SK_ID_PREV')
if 'SK_ID_CURR' in num_cols_ins:
    num_cols_ins.remove('SK_ID_CURR')

installments_payments_agg_dict = {}
for col in num_cols_ins:
   installments_payments_agg_dict[col] = ["min",'max','std','mean']

installments_payments_merge = installments_payments.groupby('SK_ID_CURR').agg(installments_payments_agg_dict).reset_index()

installments_payments_merge.columns = ['_'.join(col).strip() if col[0] != 'SK_ID_CURR' else col[0] for col in installments_payments_merge.columns.values]
installments_payments_merge.head()

,SK_ID_CURR,NUM_INSTALMENT_VERSION_min,NUM_INSTALMENT_VERSION_max,NUM_INSTALMENT_VERSION_std,NUM_INSTALMENT_VERSION_mean,NUM_INSTALMENT_NUMBER_min,NUM_INSTALMENT_NUMBER_max,NUM_INSTALMENT_NUMBER_std,NUM_INSTALMENT_NUMBER_mean,DAYS_INSTALMENT_min,...,DAYS_ENTRY_PAYMENT_std,DAYS_ENTRY_PAYMENT_mean,AMT_INSTALMENT_min,AMT_INSTALMENT_max,AMT_INSTALMENT_std,AMT_INSTALMENT_mean,AMT_PAYMENT_min,AMT_PAYMENT_max,AMT_PAYMENT_std,AMT_PAYMENT_mean
0,100001,1.0,2.0,0.377964,1.142857,1,4,1.112697,2.714286,-2916.0,...,643.904237,-2195.000000,3951.000,17397.900,5076.676624,5885.132143,3951.000,17397.900,5076.676624,5885.132143
1,100002,1.0,2.0,0.229416,1.052632,1,19,5.627314,10.000000,-565.0,...,172.058877,-315.421053,9251.775,53093.745,10058.037722,11559.247105,9251.775,53093.745,10058.037722,11559.247105
2,100003,1.0,2.0,0.200000,1.040000,1,12,3.134751,5.080000,-2310.0,...,757.325432,-1385.320000,6662.970,560835.360,110542.592300,64754.586000,6662.970,560835.360,110542.592300,64754.586000
3,100004,1.0,2.0,0.577350,1.333333,1,3,1.000000,2.000000,-784.0,...,34.019602,-761.666667,5357.250,10573.965,3011.871810,7096.155000,5357.250,10573.965,3011.871810,7096.155000
4,100005,1.0,2.0,0.333333,1.111111,1,9,2.738613,5.000000,-706.0,...,90.554005,-609.555556,4813.200,17656.245,4281.015000,6240.205000,4813.200,17656.245,4281.015000,6240.205000


In [27]:
merged_bur_credit_train = pd.merge(merged_bureau_train ,credut_balance_merge,on = 'SK_ID_CURR' , how = 'left')
merged_bur_credit_test= pd.merge(merged_bureau_test ,credut_balance_merge,on = 'SK_ID_CURR' , how = 'left')

In [28]:
merged_bur_credit_ins_train = pd.merge(merged_bur_credit_train ,installments_payments_merge,on = 'SK_ID_CURR' , how = 'left')
merged_bur_credit_ins_test= pd.merge(merged_bur_credit_test ,installments_payments_merge,on = 'SK_ID_CURR' , how = 'left')

In [29]:
pos_cash

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0
...,...,...,...,...,...,...,...,...
10001353,2448283,226558,-20,6.0,0.0,Active,843,0
10001354,1717234,141565,-19,12.0,0.0,Active,602,0
10001355,1283126,315695,-21,10.0,0.0,Active,609,0
10001356,1082516,450255,-22,12.0,0.0,Active,614,0


In [30]:
num_cols_pos = pos_cash.select_dtypes(include= [np.number]).columns.tolist()

num_cols_pos.remove('SK_ID_PREV')
if 'SK_ID_CURR' in num_cols_pos:
    num_cols_pos.remove('SK_ID_CURR')

pos_cash_agg_dict = {}
for col in num_cols_pos:
   pos_cash_agg_dict[col] = ["min",'max','std','mean']

pos_cash_merge = pos_cash.groupby('SK_ID_CURR').agg(pos_cash_agg_dict).reset_index()

pos_cash_merge.columns = ['_'.join(col).strip() if col[0] != 'SK_ID_CURR' else col[0] for col in pos_cash_merge.columns.values]
pos_cash_merge.head()

,SK_ID_CURR,MONTHS_BALANCE_min,MONTHS_BALANCE_max,MONTHS_BALANCE_std,MONTHS_BALANCE_mean,CNT_INSTALMENT_min,CNT_INSTALMENT_max,CNT_INSTALMENT_std,CNT_INSTALMENT_mean,CNT_INSTALMENT_FUTURE_min,...,CNT_INSTALMENT_FUTURE_std,CNT_INSTALMENT_FUTURE_mean,SK_DPD_min,SK_DPD_max,SK_DPD_std,SK_DPD_mean,SK_DPD_DEF_min,SK_DPD_DEF_max,SK_DPD_DEF_std,SK_DPD_DEF_mean
0,100001,-96,-53,20.863312,-72.555556,4.0,4.0,0.000000,4.000000,0.0,...,1.424001,1.444444,0,7,2.333333,0.777778,0,7,2.333333,0.777778
1,100002,-19,-1,5.627314,-10.000000,24.0,24.0,0.000000,24.000000,6.0,...,5.627314,15.000000,0,0,0.000000,0.000000,0,0,0.000000,0.000000
2,100003,-77,-18,24.640162,-43.785714,6.0,12.0,2.806597,10.107143,0.0,...,3.842811,5.785714,0,0,0.000000,0.000000,0,0,0.000000,0.000000
3,100004,-27,-24,1.290994,-25.500000,3.0,4.0,0.500000,3.750000,0.0,...,1.707825,2.250000,0,0,0.000000,0.000000,0,0,0.000000,0.000000
4,100005,-25,-15,3.316625,-20.000000,9.0,12.0,0.948683,11.700000,0.0,...,3.614784,7.200000,0,0,0.000000,0.000000,0,0,0.000000,0.000000


In [31]:
master_train = pd.merge(merged_bur_credit_ins_train ,pos_cash_merge,on = 'SK_ID_CURR' , how = 'left')
master_test= pd.merge(merged_bur_credit_ins_test ,pos_cash_merge,on = 'SK_ID_CURR' , how = 'left')

In [32]:
# حفظ الداتا بصيغة باركيه السريعة جداً
master_train.to_parquet('../data/master_train.parquet')
master_test.to_parquet('../data/master_test.parquet')

print("Data saved successfully!")

Data saved successfully!
